# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata and display summary
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata
print(f"Dataset Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs according to the Croissant schema.

All schema elements are referenced via their unique `@id` as per best practices.

In [ ]:
from pprint import pprint

# List all record sets in the metadata
record_sets = []
if hasattr(metadata, 'record_set'):
    if isinstance(metadata.record_set, list):
        record_sets = metadata.record_set
    elif metadata.record_set is not None:
        record_sets = [metadata.record_set]

print("Available Record Set `@id`s:")
for rec in record_sets:
    print(f"- {getattr(rec, '@id', rec)}")

# For each Record Set, print basic info and its fields
for rec in record_sets:
    print(f"\nRecord Set: {getattr(rec, '@id', 'N/A')}")
    print(f"  Name: {getattr(rec, 'name', 'N/A')}")
    fields = []
    if hasattr(rec, 'field'):
        flds = rec.field if isinstance(rec.field, list) else [rec.field]
        print("  Fields and their @id:")
        for field in flds:
            print(f"    - {getattr(field, 'name', 'N/A')} (@id: {getattr(field, '@id', 'N/A')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the overview steps above.

In [ ]:
# Collect record set @ids
record_set_ids = []
for rec in record_sets:
    rs_id = getattr(rec, '@id', None)
    if rs_id:
        record_set_ids.append(rs_id)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(ds.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for Record Set {record_set_id}.")

# If dataset only contains one record set (typical for tabular data), display its columns and a sample
if record_set_ids:
    main_rs_id = record_set_ids[0]
    columns = dataframes[main_rs_id].columns.tolist()
    print(f"\nColumns in record set {main_rs_id}:")
    print(columns)
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

We'll demonstrate filtering and normalization using the `gad7_score` field if it exists. All field/column references are by `@id`/column name.

In [ ]:
# Find a likely numeric field
import numpy as np

# Choose the first record set
eda_rs_id = main_rs_id if record_set_ids else None
df = dataframes[eda_rs_id].copy() if eda_rs_id else pd.DataFrame()

# Find possible numeric fields (typical for GAD-7, PHQ-9, PSQ, etc.)
possible_numeric = [col for col in df.columns if 'score' in col or 'gad7' in col or 'phq9' in col or 'psq' in col]
print(f"Possible numeric fields: {possible_numeric}")
numeric_field = None
if possible_numeric:
    # Try to pick 'gad7_score' if present, else default to first numeric
    for cand in possible_numeric:
        if 'gad' in cand:
            numeric_field = cand
            break
    else:
        numeric_field = possible_numeric[0]

if numeric_field is None and len(df.columns) > 0:
    # As a fallback, take the first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

print(f"Using numeric field for EDA: {numeric_field}")
if numeric_field is None or df.empty:
    print("Could not find a suitable numeric field.\n")
else:
    # Basic stats
    print(df[numeric_field].describe())
    
    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records with {numeric_field} > {threshold} (75th percentile):")
    print(filtered_df.head())
    
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping, e.g., by gender if present
    group_field = None
    for cand in ['gender', 'sex', 'participant_gender']:
        if cand in df.columns:
            group_field = cand
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df)
    else:
        print("No grouping field like 'gender' found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using popular plotting libraries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
if numeric_field and not df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15, color='steelblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If group_field is available, show boxplot
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook we demonstrated how to load, explore, and analyze the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, using the `mlcroissant` library. We walked through dataset metadata, examined schema structure by traversing record sets and field `@id`s, loaded and processed records into pandas DataFrames, and performed basic exploratory analysis and visualizations.

To extend this analysis, you could:
- Explore other fields (e.g., PHQ-9 or PSQ scores) referenced by their `@id`
- Examine demographic patterns by grouping and aggregating using `@id`/column names
- Use FAIR principles to integrate with external datasets.

For official documentation and advanced use cases, see: [mlcroissant documentation](https://github.com/mlcommons/croissant)
